In [1]:
import torch
import math
import torch.nn as nn

In [2]:
class ScratchEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx=None):
        super().__init__()

        #creating embedding matrics
        self.embedding_matrix = nn.Parameter(
            torch.randn(vocab_size, embed_dim)
        )

        self.embed_dim = embed_dim
        self.padding_idx = padding_idx


    def forward(self,x):
        out = self.embedding_matrix[x]

        if self.padding_idx is not None:
            mask = (x == self.padding_idx).unsqueeze(-1)
            out = out.masked_fill(mask, 0)
        out = out * math.sqrt(self.embed_dim)   # scaling

            
        return out

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len, padding_idx=None):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)

        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2) * (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        self.padding_idx = padding_idx
    
    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

In [4]:
class FeedForwardNN(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)

        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, S, D = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        weights = torch.softmax(scores, dim=-1)
        out = torch.matmul(weights, V)

        # Combine heads
        out = out.transpose(1, 2).contiguous().view(B, S, D)

        out = self.fc_out(out)

        return out


In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = FeedForwardNN(embed_dim, ff_dim)

    def forward(self, x):
    # Self-attention
        attn_out = self.attention(x)

        # Add & Norm
        x = self.norm1(x + attn_out)

        ff_out = self.ffn(x)

        # Add & Norm
        x = self.norm2(x + ff_out)

        return x

In [7]:
class TransformerInputEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim,ff_dim, num_heads, padding_idx=None):
        super().__init__()

        self.embedding = ScratchEmbedding(vocab_size, embed_dim, padding_idx)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len=5000)
        self.encoder = EncoderBlock(embed_dim, num_heads, ff_dim)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.encoder(x) 

        return x

In [ ]:
if __name__ == "__main__":
    vocab_size = 10000
    embed_dim = 512
    num_heads = 8
    ff_dim = 2048
    batch_size = 2
    seq_len = 5

    model = TransformerInputEmbedding(vocab_size, embed_dim,ff_dim, num_heads,padding_idx=0)
    x = torch.randint(0, vocab_size, (batch_size, seq_len))

    output = model(x)

    print("Input shape:", x.shape)
    print("Output shape:", output.shape)

Input shape: torch.Size([2, 5])
Output shape: torch.Size([2, 5, 512])
